# Training Dataset Preparation

**Input data:**
- `X.npy` - feature matrix (rows x 46 features)
- `y.npy` - class labels
- `file_boundaries.npy` - boundaries of each file in X
- `file_names.json` - file names

**Output data:**
- `scaler.pkl` - trained StandardScaler

**Step order:**
1. Load data
2. Split files 70/15/15
3. Fit StandardScaler on label=0 from train
4. Transform train/val/test
5. Slice into sliding windows without crossing file boundaries
6. Save

In [ ]:
import json
import pickle
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

In [ ]:
# Configuration
DATA_DIR = Path('../data')

# Sliding-window parameters
# 1 phasor = 80 samples = 20 ms (one 50 Hz period)
WINDOW_SIZE = 10   # window (10 phasors = 10 × 20 ms = 200 ms)
STEP        = 5    # step  (5 phasors =  5 × 20 ms =  100 ms)
HORIZON     = 0    # prediction horizon

# Split ratios
TRAIN_RATIO = 0.7
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

RANDOM_SEED = 42

## Load data

In [ ]:
print('Loading data...')
X               = np.load(DATA_DIR / 'X.npy')
y               = np.load(DATA_DIR / 'y.npy')
file_boundaries = np.load(DATA_DIR / 'file_boundaries.npy')

with open(DATA_DIR / 'file_names.json', encoding='utf-8') as f:
    file_names = json.load(f)

print(f'X.shape:          {X.shape}')
print(f'y.shape:          {y.shape}')
print(f'Files:           {len(file_names)}')
print(f'Balance: normal mode={(y==0).sum()}  fault mode={(y==1).sum()}  ({(y==1).mean()*100:.1f}% faults)')

## Train/val/test split

In [ ]:
n_files = len(file_names)
indices = np.arange(n_files)

# Shuffle with a fixed seed for reproducibility
rng = np.random.default_rng(RANDOM_SEED)
rng.shuffle(indices)

n_train = int(n_files * TRAIN_RATIO)
n_val   = int(n_files * VAL_RATIO)

train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

print(f'File split (seed={RANDOM_SEED}):')
print(f'  Train: {len(train_idx)} files  ({len(train_idx)/n_files*100:.1f}%)')
print(f'  Val:   {len(val_idx)} files  ({len(val_idx)/n_files*100:.1f}%)')
print(f'  Test:  {len(test_idx)} files  ({len(test_idx)/n_files*100:.1f}%)')

In [ ]:
def collect_rows(file_indices):
    """Collects X and y rows by file indices using file_boundaries."""
    X_parts, y_parts = [], []
    for i in file_indices:
        start, end = file_boundaries[i]
        X_parts.append(X[start:end])
        y_parts.append(y[start:end])
    return np.concatenate(X_parts), np.concatenate(y_parts)

X_train_raw, y_train_raw = collect_rows(train_idx)
X_val_raw,   y_val_raw   = collect_rows(val_idx)
X_test_raw,  y_test_raw  = collect_rows(test_idx)

print(f'Sizes before normalization:')
print(f'  Train: {X_train_raw.shape}  fault: {(y_train_raw==1).mean()*100:.1f}%')
print(f'  Val:   {X_val_raw.shape}    fault: {(y_val_raw==1).mean()*100:.1f}%')
print(f'  Test:  {X_test_raw.shape}   fault: {(y_test_raw==1).mean()*100:.1f}%')

## Fit StandardScaler on label=0 from train

The scaler is trained only on normal operating mode (label=0) from the training set.
This ensures that:
- features in normal mode are close to 0
- during faults, features deviate sharply from 0
- the scaler is not contaminated by anomalous values during faults

In [ ]:
# English note.
# X_train_normal = X_train_raw[y_train_raw == 0]
# print(f'  Normal-mode rows: {len(X_train_normal)}')

# scaler = StandardScaler()
# scaler.fit(X_train_normal)
# print('  Done!')

## Transform train/val/test

In [ ]:
# X_train_scaled = scaler.transform(X_train_raw).astype(np.float32)
# X_val_scaled   = scaler.transform(X_val_raw).astype(np.float32)
# X_test_scaled  = scaler.transform(X_test_raw).astype(np.float32)

# English note.
# print(f'  mean: {X_train_scaled[y_train_raw==0].mean():.4f}  (should be ~0)')
# print(f'  std:  {X_train_scaled[y_train_raw==0].std():.4f}   (should be ~1)')

X_train_scaled = X_train_raw
X_val_scaled   = X_val_raw
X_test_scaled  = X_test_raw

## Sliding Window Slicing

In [ ]:
def make_windows(X_scaled, y_raw, file_indices, window, step, horizon=0):
    """Slices into windows while respecting file boundaries.
    Window label = 1 if at least one phasor in the window is labeled as a fault."""
    all_X, all_y = [], []
    cursor = 0
    for i in file_indices:
        start, end = file_boundaries[i]
        n_rows = end - start
        X_file = X_scaled[cursor:cursor + n_rows]
        y_file = y_raw[cursor:cursor + n_rows]

        limit = n_rows - window - horizon
        for s in range(0, limit + 1, step):
            all_X.append(X_file[s:s + window])
            all_y.append(int(y_file[s:s + window].any()))
        cursor += n_rows
    return np.array(all_X, dtype=np.float32), np.array(all_y, dtype=np.int64)

X_train, y_train = make_windows(X_train_scaled, y_train_raw, train_idx, WINDOW_SIZE, STEP, HORIZON)
X_val,   y_val   = make_windows(X_val_scaled,   y_val_raw,   val_idx,   WINDOW_SIZE, STEP, HORIZON)
X_test,  y_test  = make_windows(X_test_scaled,  y_test_raw,  test_idx,  WINDOW_SIZE, STEP, HORIZON)

print(f'\nShapes after slicing:')
print(f'  X_train: {X_train.shape}  fault: {(y_train==1).mean()*100:.1f}%')
print(f'  X_val:   {X_val.shape}    fault: {(y_val==1).mean()*100:.1f}%')
print(f'  X_test:  {X_test.shape}   fault: {(y_test==1).mean()*100:.1f}%')

## Save

In [ ]:
np.save(DATA_DIR / 'X_train.npy', X_train)
np.save(DATA_DIR / 'X_val.npy',   X_val)
np.save(DATA_DIR / 'X_test.npy',  X_test)
np.save(DATA_DIR / 'y_train.npy', y_train)
np.save(DATA_DIR / 'y_val.npy',   y_val)
np.save(DATA_DIR / 'y_test.npy',  y_test)

# with open(DATA_DIR / 'scaler.pkl', 'wb') as f:
#     pickle.dump(scaler, f)

print(f'  X_train.npy  {X_train.shape}')
print(f'  X_val.npy    {X_val.shape}')
print(f'  X_test.npy   {X_test.shape}')
print(f'  y_train.npy  {y_train.shape}')
print(f'  y_val.npy    {y_val.shape}')
print(f'  y_test.npy   {y_test.shape}')